In [12]:
import sys
sys.path.append('..') 

import logging
from sqlalchemy import create_engine, text

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import API_AMIGOCLOUD_TOKEN_ADM
from config import POSTGRES_UTEA

In [13]:
# Configurar logging para ver el progreso paso a paso en consola
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def obtener_engine():
    return create_engine(
        f"postgresql+psycopg2://{POSTGRES_UTEA['USER']}:{POSTGRES_UTEA['PASSWORD']}@{POSTGRES_UTEA['HOST']}:{POSTGRES_UTEA['PORT']}/{POSTGRES_UTEA['DATABASE']}"
    )

def procesar_catastro():
    engine = obtener_engine()
    
    # 1. Consulta SQL para resetear los valores actuales del catastro
    sql_reset = """
    UPDATE catastro_iag.catastro 
    SET porcen_cos = 0, 
        fc = NULL;
    """
    
    # 2. Consulta SQL con la conversión de metros cuadrados a hectáreas ( / 10000 )
    sql_update = """
    WITH calculo_cosecha AS (
        SELECT 
            c.idd,
            -- Convertimos ST_Area (m2) a Hectáreas dividiendo entre 10000.
            -- Luego sumamos, dividimos entre el área del catastro (ha) y multiplicamos por 100 para el %.
            COALESCE(
                SUM(ST_Area(ST_Intersection(c.geom, cos.geom)) / 10000.0) / NULLIF(c.area, 0) * 100, 
                0
            ) AS porcentaje_calculado,
            -- Extraemos la fecha de cosecha (fc) más reciente (la mayor)
            MAX(cos.fc) AS max_fc
        FROM catastro_iag.catastro c
        -- Unimos espacialmente con la capa de cosechas
        INNER JOIN catastro_iag.cosecha_2026 cos 
            ON ST_Intersects(c.geom, cos.geom)
        WHERE 
            cos.fc IS NOT NULL 
            AND cos.estado = 'Cosechado'
        GROUP BY c.idd, c.area
    )
    UPDATE catastro_iag.catastro orig
    SET 
        porcen_cos = cc.porcentaje_calculado,
        fc = cc.max_fc
    FROM calculo_cosecha cc
    WHERE orig.idd = cc.idd;
    """
    
    try:
        # 'engine.begin()' abre una transacción automática
        with engine.begin() as conexion:
            
            logging.info("Paso 1: Reiniciando valores (porcen_cos = 0 y fc = NULL) en la tabla catastro...")
            res_reset = conexion.execute(text(sql_reset))
            logging.info(f"Valores reiniciados en {res_reset.rowcount} registros.")
            
            logging.info("Paso 2: Calculando áreas en hectáreas y actualizando datos con PostGIS...")
            res_update = conexion.execute(text(sql_update))
            logging.info(f"¡Proceso completado! Se actualizaron {res_update.rowcount} lotes con datos de cosecha consistentes en hectáreas.")
            
    except Exception as e:
        logging.error(f"Error crítico durante el procesamiento: {e}")
        raise e

In [11]:
if __name__ == "__main__":
    procesar_catastro()

2026-07-12 16:22:06,843 - INFO - Paso 1: Reiniciando valores (porcen_cos = 0 y fc = NULL) en la tabla catastro...
2026-07-12 16:22:07,020 - INFO - Valores reiniciados en 14566 registros.
2026-07-12 16:22:07,021 - INFO - Paso 2: Calculando áreas en hectáreas y actualizando datos con PostGIS...
2026-07-12 16:22:15,550 - INFO - ¡Proceso completado! Se actualizaron 2319 lotes con datos de cosecha consistentes en hectáreas.
